# MAC Toy Model: Labels, Clearances, and Compartments


## Teaching goal

This is a toy model of Mandatory Access Control (MAC). It compares subject clearances with object
labels. Real MAC systems are implemented by operating systems, database engines, or specialized
platforms. Administrators choose labels, compartments, and policy rules.


In [ ]:
from dataclasses import dataclass
from pprint import pprint


@dataclass(frozen=True)
class Request:
    user: str
    operation: str
    resource: str
    context: dict


users = {
    "dr_rossi": {"name": "Dr. Rossi", "department": "cardiology"},
    "nurse_amina": {"name": "Nurse Amina", "department": "ward-a"},
    "lab_tech": {"name": "Lab Technician", "department": "laboratory"},
    "billing_clerk": {"name": "Billing Clerk", "department": "billing"},
    "privacy_auditor": {"name": "Privacy Auditor", "department": "compliance"},
}

resources = {
    "ehr_note_42": {"type": "ehr_note", "patient": "patient-42", "department": "cardiology"},
    "lab_result_42": {"type": "lab_result", "patient": "patient-42", "department": "laboratory"},
    "billing_record_42": {"type": "billing_record", "patient": "patient-42", "department": "billing"},
    "audit_log": {"type": "audit_log", "patient": None, "department": "compliance"},
}

requests = [
    Request("dr_rossi", "read", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("nurse_amina", "write", "ehr_note_42", {"assigned_patient": True, "emergency": False}),
    Request("lab_tech", "write", "lab_result_42", {"assigned_patient": False, "emergency": False}),
    Request("billing_clerk", "read", "ehr_note_42", {"assigned_patient": False, "emergency": False}),
    Request("privacy_auditor", "read", "audit_log", {"assigned_patient": False, "emergency": False}),
]


In [ ]:
# Rank increases with sensitivity.
ranks = {
    "public": 0,
    "internal": 1,
    "confidential": 2,
    "restricted": 3,
}

# A clearance combines a rank with the compartments the subject belongs to.
clearances = {
    "dr_rossi": {"rank": "restricted", "compartments": {"cardiology"}},
    "nurse_amina": {"rank": "confidential", "compartments": {"ward-a", "cardiology"}},
    "lab_tech": {"rank": "confidential", "compartments": {"laboratory"}},
    "billing_clerk": {"rank": "internal", "compartments": {"billing"}},
    "privacy_auditor": {"rank": "restricted", "compartments": {"compliance", "cardiology", "laboratory"}},
}

# A label defines the sensitivity and compartment required by the resource.
labels = {
    "ehr_note_42": {"rank": "confidential", "compartments": {"cardiology"}},
    "lab_result_42": {"rank": "confidential", "compartments": {"laboratory"}},
    "billing_record_42": {"rank": "internal", "compartments": {"billing"}},
    "audit_log": {"rank": "restricted", "compartments": {"compliance"}},
}


In [ ]:
def dominates(clearance: dict, label: dict) -> bool:
    # Rank must be high enough.
    rank_ok = ranks[clearance["rank"]] >= ranks[label["rank"]]

    # The subject must have every compartment required by the object.
    compartments_ok = label["compartments"].issubset(clearance["compartments"])

    return rank_ok and compartments_ok


def mac_allows(request: Request) -> bool:
    # This toy rule focuses on read access; real MAC models also constrain writes.
    if request.operation != "read":
        return False
    return dominates(clearances[request.user], labels[request.resource])


for request in requests:
    print(request.user, request.operation, request.resource, "=>", mac_allows(request))


In [ ]:
# Changing a label changes the decision even when user identity stays the same.
request = Request("dr_rossi", "read", "ehr_note_42", {})
print("Before relabel:", mac_allows(request))

labels["ehr_note_42"] = {"rank": "restricted", "compartments": {"psychiatry"}}
print("After relabel:", mac_allows(request))

# Restore the original label for later experimentation.
labels["ehr_note_42"] = {"rank": "confidential", "compartments": {"cardiology"}}


## What to notice

MAC is less about personal sharing and more about centrally enforced information flow. A user with a
valid account and a senior job title can still be denied if the clearance does not dominate the
resource label.
